# 🦜 Personal Resource Assistant — RAG with LangChain, OpenAI & ChromaDB — Explained

**What this notebook does:** the exact same RAG (Retrieval-Augmented Generation) assistant as `RAG_Personal_Resource_Assistant_Langchain,_Cohere_and_ChromaDB.ipynb`, rewritten to use the **OpenAI** ecosystem instead of Cohere — `OpenAIEmbeddings` for retrieval and `ChatOpenAI` for generation, both from `langchain-openai`. Everything else — PDF extraction, chunking, ChromaDB, the LCEL pipeline, and `.invoke()` / `.batch()` / `.stream()` — stays conceptually identical. Each code cell below has a short explanation directly above it.

## Installing necessary libraries

OpenAI does not offer a free trial tier the way Cohere does — you'll need a funded API key. Generate one from [platform.openai.com/api-keys](https://platform.openai.com/api-keys).

### 📦 Cell — Install the libraries

Same as the Cohere version, but `langchain-cohere` is swapped for **`langchain-openai`** — LangChain's integration package for OpenAI's chat and embedding models. Everything else (`pdfminer.six` for PDFs, `chromadb` for the vector store, the text splitter) is unchanged, since none of that is provider-specific.

In [ ]:
!pip install langchain-openai langchain pdfminer.six chromadb langchain-community langchain-text-splitters

langchain-openai: Enables integration of OpenAI's language and embedding models with LangChain for advanced text generation and processing workflows.

langchain: Provides a modular framework for building language model-powered applications, such as chatbots, question-answering systems, and conversational agents.

pdfminer.six: Facilitates text extraction from PDF files, making it useful for document analysis and preprocessing tasks.

chromadb: A vector database library designed for efficient storage and retrieval of embeddings, ideal for tasks like semantic search and recommendation systems.

## Importing libraries

### 🧰 Cell — Import everything, and set the OpenAI API key

Reads the OpenAI API key from Colab's secret storage and sets it as an environment variable — `ChatOpenAI` and `OpenAIEmbeddings` both pick this up automatically, the same pattern `ChatCohere` and `CohereEmbeddings` used for `COHERE_API_KEY`. Everything else is the same import list as before, just with `langchain_openai` swapped in for `langchain_cohere`.

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from pdfminer.high_level import extract_text as extract_text_pdf_miner
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

## VectorDB setup

### 🗄️ Cell — Set up where the vector database lives, and which embedding model to use

`persist_directory` works exactly the same as the Cohere version — the folder on disk where Chroma saves its data, so it survives a restart. `OpenAIEmbeddings` replaces `CohereEmbeddings`, using `text-embedding-3-small` — OpenAI's current, cost-efficient embedding model. As always, this exact same model has to embed both the PDF chunks *and* every question later, or the comparisons wouldn't mean anything.

In [ ]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/chroma_db_openai"

# Initialize OpenAI embeddings with the specified model
# "text-embedding-3-small" is OpenAI's current, cost-efficient embedding model
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

We are processing 2 research papers on transformers and yolo. You can use your own PDFs.

### 📄 Cell — Turn each PDF into chunks, and store them in the vector database

Identical to the Cohere version, and deliberately so — chunking and storage have nothing to do with which model provider you use. Each PDF's text is extracted, cleaned, split into 2048-character chunks with 512 characters of overlap, wrapped in `Document` objects tagged with their source file, then embedded (this time by OpenAI) and saved into the persistent Chroma database.

In [ ]:
# Loop through a list of PDF files to process
for pdf_name in ["/content/1706.03762v7.pdf", "/content/1506.02640v5.pdf"]:
    # Open each PDF file in binary mode
    with open(pdf_name, 'rb') as f:
        # Extract text from the PDF using the extract_text_pdf_miner function
        text = extract_text_pdf_miner(f)

        # Clean the extracted text by removing newline characters and joining into a single string
        cleaned_text = " ".join(text.split("\n"))

        # Initialize a list to store document chunks
        docs = []

        # Create a text splitter to divide the text into manageable chunks
        # Each chunk has a maximum size of 2048 characters with a 512-character overlap
        splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

        # Split the cleaned text into chunks and wrap each chunk in a Document object
        for chunk in splitter.split_text(cleaned_text):
            docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))

    # Create a Chroma collection from the processed documents
    # Use the specified persist directory and embedding model for storage and retrieval
    vector_collection_fixed_size = Chroma.from_documents(
        documents=docs,
        persist_directory=persist_directory,
        embedding=embedding
    )

### 🔌 Cell — Reconnect to the vector database

Opens a handle to the same persistent Chroma database just filled in the previous cell — same `persist_directory`, same OpenAI embedding model.

In [ ]:
# Initialize a Chroma vector database
# The persist_directory specifies the location where the database is stored
# The embedding_function parameter provides the embedding model used for vector representation
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

### 🔍 Cell — Try a similarity search directly

Same sanity check as before: search the vector database directly for the single closest chunk (`k=1`) to “What is YOLO?”, with a relevance score — dense vector retrieval on its own, no LLM involved yet.

In [ ]:
# Perform a similarity search on the vector database
# The query "What is YOLO?" is used to find the most relevant documents
# k=1 specifies that the top 1 most similar document should be retrieved
# The method also returns relevance scores indicating how closely each document matches the query
vectordb.similarity_search_with_relevance_scores("What is YOLO?", k=1)

## RAG pipeline

### 🔗 Cell — Build the full RAG pipeline with LCEL

The same pipeline shape as the Cohere version, with the model swapped:

- **`llm`** — `ChatOpenAI` using `gpt-4o-mini`, with `temperature=0` for consistent, deterministic answers.
- **`prompt`** — the identical two-placeholder template (`{context}`, `{question}`), instructing the model to answer strictly from the given context.
- **`retrieval`** — unchanged: `RunnableParallel` runs `vectordb.as_retriever()` (fetch relevant chunks) and `RunnablePassthrough()` (keep the question as-is) side by side, producing `{"context": ..., "question": ...}`.
- **`chain = retrieval | prompt | llm | output_parser`** — the exact same LCEL pipe shape as the Cohere notebook; only the model inside it changed.

**Example:** ask “What is YOLO?” — the retrieved chunks and the question get woven into the prompt, `gpt-4o-mini` drafts an answer grounded in that context, and the parser hands back plain text — the provider changed, the shape of the pipeline didn't.

In [ ]:
# Initialize an LLM instance using OpenAI's "gpt-4o-mini" model
# The temperature parameter controls randomness in the generated responses; 0 ensures deterministic outputs
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define a prompt template for generating answers based on a given context and question
prompt_str = """Answer the question below using the context:

Context: {context}

Question: {question}

Answer: """

# Create a ChatPromptTemplate from the string template, enabling dynamic input for context and question
prompt = ChatPromptTemplate.from_template(prompt_str)

# Create a retrieval pipeline to fetch relevant context and pass through the user's question
retrieval = RunnableParallel(
    {
        # Use the vector database as a retriever to fetch relevant context for the question
        "context": vectordb.as_retriever(),

        # Pass through the user's input question without modification
        "question": RunnablePassthrough()
    }
)

# Define an output parser to format the generated response into a string
output_parser = StrOutputParser()

# Create a processing chain that retrieves context, formats the prompt, generates an LLM response, and parses the output
chain = retrieval | prompt | llm | output_parser

### ▶️ Cell — Run the full RAG pipeline

Same as before — `.invoke(...)` runs the entire chain end to end and returns just the final text.

In [ ]:
# Invoke the chain of components (retrieval, prompt generation, LLM processing, and output parsing)
# The question "What is YOLO?" is passed through the chain to generate the response
response = chain.invoke("What is YOLO?")

# Print the response generated by the chain
print(response)

## Other chain invoking methods!

.invoke(): The goal is to pass in an input and receive the output—neither more nor less.

.batch(): This is faster than using invoke three times when you wish to supply several inputs to get multiple outputs because it handles the parallelization for you.

.stream():  We may begin printing the response before the entire response is complete.

### 📚 Cell — Run the chain on multiple questions at once

Unchanged from the Cohere version — `.batch([...])` runs the same chain over several questions in one call, returning a list of answers in the same order.

In [ ]:
response_with_batch = chain.batch(["What is Transformers", "How is Transformer different than YOLO?"])

for response in response_with_batch:
  print(response)
  print("\n")

### 🌊 Cell — Stream the answer as it's generated

Also unchanged — `.stream(...)` yields pieces of the answer as `gpt-4o-mini` generates them, printed with `end=""` so the words appear one after another instead of all at once.

In [ ]:
for chunk in chain.stream("What are the 3 vectors in Transformers architecture?"):
  print(chunk, flush=True, end="")